### Тестирование модели TSMIXER с экзогенными переменными

Исходные данные - часовые данные акций SBER

In [1]:
#!pip install neuralforecast vectorbt scikit-learn mlflow plotly pandas nbformat

In [2]:
import torch
torch.cuda.is_available()

True

In [3]:
torch.cuda.empty_cache()

In [4]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from neuralforecast import NeuralForecast
from neuralforecast.auto import AutoTSMixerx

import mlflow
import plotly.express as px
import plotly.graph_objects as go
from neuralforecast.losses.pytorch import MQLoss, MASE, MAE
from utilsforecast.plotting import plot_series
from utilsforecast.preprocessing import fill_gaps

In [5]:
%run ../../base/set_secrets.ipynb
%run ../../base/plot.ipynb
%run ../../prepare_data/prepare_data.ipynb

In [6]:
ClassModel = AutoTSMixerx
model_name = 'AutoTSMixerx'
data_interval = '1hour'
experiment = 'TSMixerx_2' + data_interval
horizon = 12  

In [7]:
# Загрузка и подготовка данных
df = pd.read_csv('../../data/SBER/'+data_interval+'.csv')
df.rename(columns={'time': 'Datetime', 'open':'Open','close':'Close', 'high':'High', 'low':'Low', 'volume':'Volume'}, inplace=True)

# create_features из prepare_data
df, new_columns = prepare_data(df.copy(), [])

len(df)

27471

In [8]:
fig_candlestick = create_candlestick_chart(df.tail(100*24), title="SBER")
fig_candlestick.show()


In [9]:
# Подготавливаем данные для NeuralForecast
df = df.rename(columns={'Datetime': 'ds', 'Close': 'y'})
df['ds'] = pd.to_datetime(df['ds'])
df['unique_id'] = 1

df['y'] = df['y'].interpolate(method='linear', limit_direction='both')


In [10]:
# Настройка MLflow
try:
    mlflow.set_tracking_uri("http://localhost:8080")
    mlflow.set_experiment(experiment)
    print("MLflow успешно подключен")
except Exception as e:
    print(f"Ошибка при настройке MLflow: {e}")

MLflow успешно подключен


In [11]:
# Разделение данных на тренировочную и тестовую выборки
test_size = 90 * 24  # 90 дней 
train = df.iloc[:-test_size]
test = df.iloc[-test_size:]

print(f'Train size: {len(train)}')
print(f'Test size: {len(test)}')
print(f'Train period: {train["ds"].min()} - {train["ds"].max()}')
print(f'Test period: {test["ds"].min()} - {test["ds"].max()}')

Train size: 25311
Test size: 2160
Train period: 2018-03-08 07:00:00+00:00 - 2024-12-09 15:00:00+00:00
Test period: 2024-12-09 16:00:00+00:00 - 2025-04-04 12:00:00+00:00


In [12]:
# Определение экзогенных переменных
hist_exog_list=['Open', 'High', 'Low', 'Volume']
hist_exog_list = hist_exog_list + new_columns
print("Используемые экзогенные переменные:")
hist_exog_list

Используемые экзогенные переменные:


['Open',
 'High',
 'Low',
 'Volume',
 'anomalies_price',
 'anomalies_volume',
 'Open_ratio_1',
 'Open_log_diff_1',
 'Open_momentum_3',
 'Open_roc_3',
 'Open_ema_3',
 'Open_momentum_5',
 'Open_roc_5',
 'Open_ema_5',
 'Open_momentum_7',
 'Open_roc_7',
 'Open_ema_7',
 'High_ratio_1',
 'High_log_diff_1',
 'High_momentum_3',
 'High_roc_3',
 'High_ema_3',
 'High_momentum_5',
 'High_roc_5',
 'High_ema_5',
 'High_momentum_7',
 'High_roc_7',
 'High_ema_7',
 'Low_ratio_1',
 'Low_log_diff_1',
 'Low_momentum_3',
 'Low_roc_3',
 'Low_ema_3',
 'Low_momentum_5',
 'Low_roc_5',
 'Low_ema_5',
 'Low_momentum_7',
 'Low_roc_7',
 'Low_ema_7',
 'Close_ratio_1',
 'Close_log_diff_1',
 'Close_momentum_3',
 'Close_roc_3',
 'Close_ema_3',
 'Close_momentum_5',
 'Close_roc_5',
 'Close_ema_5',
 'Close_momentum_7',
 'Close_roc_7',
 'Close_ema_7',
 'Volume_ratio_1',
 'Volume_log_diff_1',
 'Volume_momentum_3',
 'Volume_roc_3',
 'Volume_ema_3',
 'Volume_momentum_5',
 'Volume_roc_5',
 'Volume_ema_5',
 'Volume_momentum_7',

In [13]:
from ray import tune

with mlflow.start_run(run_name=experiment) as run:
	
	conf=ClassModel.get_default_config(h=12, backend="optuna", n_series=1)

	config = {
		"n_series": 1,
		"input_size": tune.choice([48, 72, 96, 120]),					# Size of input window
		"learning_rate": tune.loguniform(5e-5, 5e-3),				# Initial Learning rate 
		"scaler_type": "robust",
		"n_block": tune.choice([2, 4, 6, 8, 10]),  					# Number of mixing layers
		"dropout": tune.uniform(0.1, 0.5),
		"windows_batch_size": tune.choice([128, 256, 512]),
		"max_steps": tune.choice([500, 1000, 2000]),				# Number of training iterations
		"val_check_steps": 100, 									# Compute validation every x steps
		#"early_stop_patience_steps": 5,								# Early stopping steps
		"dropout": tune.uniform(0.2, 0.8),							# Dropout
		#"ff_dim": tune.choice([32, 64, 128]),						# Dimension of the feature linear layer
		"hist_exog_list": hist_exog_list,
		"random_seed": 777,
	}

	model = ClassModel(
		config=ClassModel._ray_config_to_optuna(config),  
		n_series=1,
		h=horizon, 
		loss=MQLoss(),
		backend="optuna", 
		num_samples=25 
	)
	#model.hist_exog_list = hist_exog_list

	nf = NeuralForecast(models=[model], freq=data_interval)

	mlflow.log_params({
		'model': model_name,
		'horizon': horizon,
		'n_series': 1,
		'hist_exog': ", ".join(hist_exog_list),
		'loss': 'MQLoss',
		'backend': 'optuna',
		'num_samples': 25,
		'config' : config,
	})

	nf.fit(df=train)
	print("Обучение завершено успешно!")
	
	# Кросс-валидация с 50 окнами, step_size=horizon
	print("Старт кросс-валидации...")
	cv_results = nf.cross_validation(
	    df=df,
	    n_windows=50,
	    step_size=horizon,
	    refit=False
	)
	predict_result = model_name+"-median"
	
	# Вычисление метрик для каждого окна
	metrics_data = []
	cutoffs = cv_results['cutoff'].unique()
	for window in cutoffs:
		y_true = cv_results['y'].loc[cv_results['cutoff']==window].values		
		y_pred = cv_results[predict_result].loc[cv_results['cutoff']==window].values 
		
		mse = mean_squared_error(y_true, y_pred)
		mae = mean_absolute_error(y_true, y_pred)
		rmse = np.sqrt(mse)
		mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100 if np.any(y_true != 0) else np.nan
		
		metrics_data.append({
			'cutoff': window,
			'MSE': mse, 
			'MAE': mae,
			'RMSE': rmse,
			'MAPE': mape
		})

	metrics_df = pd.DataFrame(metrics_data)
	
	# Усреднение метрик
	avg_metrics = metrics_df.mean()
	
	# Логирование метрик в MLflow
	mlflow.log_metrics({
	    'avg_mse': avg_metrics["MSE"],
	    'avg_mae': avg_metrics["MAE"],
	    'avg_rmse': avg_metrics["RMSE"],
	    'avg_mape': avg_metrics["MAPE"]
	})
	
	# Сохранение модели в MLflow
	mlflow.pytorch.log_model(model, "model", registered_model_name="AutoTSMixerx")
	
	# Сохранение метрик в CSV 
	metrics_df.to_csv("metrics_by_window.csv", index=False)
	mlflow.log_artifact("metrics_by_window.csv")



[I 2025-04-12 19:27:17,409] A new study created in memory with name: no-name-7c5f71b6-cb59-4351-a6a1-ef5f44056bed
/home/olga/ML/Projects/Project_test/.venv/lib/python3.12/site-packages/neuralforecast/common/_base_auto.py:295: FutureWarning:

suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.

/home/olga/ML/Projects/Project_test/.venv/lib/python3.12/site-packages/neuralforecast/common/_base_auto.py:293: FutureWarning:

suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.

Seed set to 777
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                | Type              | Params | Mode 
------------------------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

`Trainer.fit` stopped: `max_steps=1000` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

2025/04/12 20:40:20 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'AutoTSMixerx' already exists. Creating a new version of this model...
2025/04/12 20:40:20 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: AutoTSMixerx, version 4


🏃 View run TSMixerx_21hour at: http://localhost:8080/#/experiments/939051360117562587/runs/fa767692f8ea475eafd12d31ae1d95e2
🧪 View experiment at: http://localhost:8080/#/experiments/939051360117562587


Created version '4' of model 'AutoTSMixerx'.


In [14]:
print(f'Average MSE: {avg_metrics["MSE"]:.4f}')
print(f'Average MAE: {avg_metrics["MAE"]:.4f}')
print(f'Average RMSE: {avg_metrics["RMSE"]:.4f}')
print(f'Average MAPE: {avg_metrics["MAPE"]:.4f}%')


Average MSE: 10.2264
Average MAE: 2.1414
Average RMSE: 2.3884
Average MAPE: 0.6865%


In [15]:
# Получение прогнозов на тестовом наборе
# predictions = nf.predict()
# predictions.head()

In [16]:
# Расчет метрик для прогнозов на тестовом наборе
# y_test = test['y'].values
# y_pred = predictions[predict_result].values

# mae = mean_absolute_error(y_test, y_pred)
# rmse = np.sqrt(mean_squared_error(y_test, y_pred))
# mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100 if np.any(y_test != 0) else np.nan

# print(f'MAE на тестовой выборке: {mae:.4f}')
# print(f'RMSE на тестовой выборке: {rmse:.4f}')
# print(f'MAPE на тестовой выборке: {mape:.4f}%')

In [17]:
# from utilsforecast.plotting import plot_series
# # для одного предсказания
# plot_series(df.iloc[:-test_size+24], predictions, plot_random=False, max_insample_length=24 * 9, engine='plotly')

In [18]:
cutoffs = cv_results['cutoff'].unique()
cutoffs

<DatetimeArray>
['2025-03-03 20:00:00+00:00', '2025-03-04 14:00:00+00:00',
 '2025-03-05 08:00:00+00:00', '2025-03-05 20:00:00+00:00',
 '2025-03-06 14:00:00+00:00', '2025-03-07 08:00:00+00:00',
 '2025-03-07 20:00:00+00:00', '2025-03-08 10:00:00+00:00',
 '2025-03-09 00:00:00+00:00', '2025-03-09 12:00:00+00:00',
 '2025-03-10 06:00:00+00:00', '2025-03-10 18:00:00+00:00',
 '2025-03-11 12:00:00+00:00', '2025-03-12 06:00:00+00:00',
 '2025-03-12 18:00:00+00:00', '2025-03-13 12:00:00+00:00',
 '2025-03-14 06:00:00+00:00', '2025-03-14 18:00:00+00:00',
 '2025-03-15 08:00:00+00:00', '2025-03-15 20:00:00+00:00',
 '2025-03-16 10:00:00+00:00', '2025-03-17 04:00:00+00:00',
 '2025-03-17 16:00:00+00:00', '2025-03-18 10:00:00+00:00',
 '2025-03-19 04:00:00+00:00', '2025-03-19 16:00:00+00:00',
 '2025-03-20 10:00:00+00:00', '2025-03-21 04:00:00+00:00',
 '2025-03-21 16:00:00+00:00', '2025-03-22 06:00:00+00:00',
 '2025-03-22 18:00:00+00:00', '2025-03-23 08:00:00+00:00',
 '2025-03-23 20:00:00+00:00', '2025-03-2

In [19]:
cutoffs = cv_results['cutoff'].unique()
lastY = df.tail(150*24)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=lastY['ds'],
    y=lastY['y'],
    mode='lines',
    name='Фактическая цена',
    line=dict(color='blue')
))

# Прогноз модели
fig.add_trace(go.Scatter(
    x=cv_results['ds'],
    y=cv_results[predict_result],
    mode='lines',
    name='Прогноз '+experiment,
    line=dict(color='red')
))

# Вертикальные линии для cutoff
for cutoff in cutoffs:
    fig.add_vline(
        x=cutoff,
        line=dict(color="black", dash="dot"),
        opacity=0.3
    )

# Настройка оформления
fig.update_layout(
    title="Результаты кросс-валидации модели",
    width=1500,
    height=600,
    xaxis_title='Дата',
    yaxis_title='Цена',
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
    margin=dict(l=20, r=20, t=50, b=20)
)

fig.show()

In [20]:
import os
# Сохранение модели
path='../../checkpoints/'+experiment+'/'
os.mkdir(path)
nf.save(path=path,
        model_index=None, 
        overwrite=True,
        save_dataset=True)
print("Модель успешно сохранена")

Модель успешно сохранена


In [21]:
cv_results

,unique_id,ds,cutoff,AutoTSMixerx-median,AutoTSMixerx-lo-90,AutoTSMixerx-lo-80,AutoTSMixerx-hi-80,AutoTSMixerx-hi-90,y
0,1,2025-03-04 03:00:00+00:00,2025-03-03 20:00:00+00:00,303.495361,299.977936,300.675568,305.290588,306.021332,306.95
1,1,2025-03-04 04:00:00+00:00,2025-03-03 20:00:00+00:00,302.687988,298.085449,299.225861,305.485474,306.151459,308.79
2,1,2025-03-04 05:00:00+00:00,2025-03-03 20:00:00+00:00,302.942139,298.428955,299.346405,305.152313,306.145233,309.35
3,1,2025-03-04 06:00:00+00:00,2025-03-03 20:00:00+00:00,302.389709,297.068207,298.427673,305.209076,306.409485,308.66
4,1,2025-03-04 07:00:00+00:00,2025-03-03 20:00:00+00:00,303.229248,298.183777,299.343872,306.130768,307.297607,312.90
...,...,...,...,...,...,...,...,...,...
595,1,2025-04-04 08:00:00+00:00,2025-04-03 18:00:00+00:00,297.985962,289.626465,291.765259,303.085754,304.390076,300.44
596,1,2025-04-04 09:00:00+00:00,2025-04-03 18:00:00+00:00,297.712097,288.964020,291.219116,302.374542,303.558655,298.27
597,1,2025-04-04 10:00:00+00:00,2025-04-03 18:00:00+00:00,297.627899,288.880737,291.037872,302.052582,303.705078,294.50
598,1,2025-04-04 11:00:00+00:00,2025-04-03 18:00:00+00:00,297.807312,288.825623,290.944580,302.378265,303.831238,294.19
